# 3DAR_EV: 위치(SSL) + 소리 종류(ID) 학습 — 한 파일 정리본

`ev3dar/` 패키지 8개 모듈과 `experiments/loc_20261116/` 실험 스크립트를 펭귄 노트북 형식으로 합친 파일이다.

| 셀 | 내용 | 펭귄 노트북 대응 |
|---|---|---|
| 1 | 설정 → 데이터 파싱/분할 → 특징 → Dataset → 모델/손실 → 지표 → 학습 루프 (seed 3개) | `3DAR_multi_localization` 셀 1 |
| 2 | test 평가, seed별 결과, 평균±표준편차, 위치 판별 지표 | 셀 2 (평가) |
| 3 | SSL 결과 그림 (펭귄 셀 2와 같은 각도-확률 그림, test 녹음마다) | 셀 2 (그림) |
| 4 | SSL + ID 합치기 → 최종 고장 위치 + 각도 | (펭귄에는 없음) |
| 5 | 규칙 기반 위치 판별 (학습 없음, 11/16 실험 1) | 셀 3의 Rule-based |
| 6 | 그림: 혼동 행렬, seed별 지표 막대 | 셀 3~4 (그림) |

- SSL은 **고장 소리 위치만** 찾는다(`ssl_target='fault'`): 고장 개수 0~3과 고장 위치 각도. Healthy는 실제 차량에서 사방에서 나는 배경이라 정답에서 뺀다.
- ID는 기본으로 위치별 **정상/고장 2종**만 판별한다(`id_task='binary'`). 6종 분류는 `id_task='type'`.
- 원본 데이터는 읽기만 한다. 결과는 `CONFIG['out_dir']`에 쓴다.
- 기본값은 기존 baseline과 같다(epoch 50, PSD/RPM 끔). 11/16 실험에서 가장 좋았던 설정은 `use_psd=True`, `use_rpm=True`(실험 3), `epochs=100`(실험 2)이다. 다만 PSD+RPM과 epoch 100을 함께 돌린 적은 아직 없다.
- 지표는 실제 예측으로 계산한다. 하드코딩한 값은 없다.
- GPU 발열을 줄이려고 `gpu_duty=0.5`로 평균 사용률을 약 50%로 제한한다(학습 시간 약 2배). 끄려면 1.0.
- 학습은 Spark GPU에서만 한다. CUDA가 없으면 멈추고, CPU로 자동 전환하지 않는다.

In [ ]:
# ==========================================
# 1. Configuration
# ==========================================
import csv, json, time, random, hashlib
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import soundfile as sf
from scipy import signal
from scipy.optimize import linear_sum_assignment
from scipy.signal import find_peaks
from sklearn.metrics import (confusion_matrix, f1_score, balanced_accuracy_score,
                             roc_auc_score, average_precision_score)
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

PKG = Path('/home/kos/Desktop/3DAR_EV/EV_3DAR_AI')

CONFIG = {
    # --- 경로 ---
    'data_root': '/home/kos/Desktop/3DAR_EV/[3DAR Electric Car]/3DAR_EV',  # 측정_목록.csv 가 있는 폴더 (읽기 전용)
    'out_dir':   str(PKG / 'runs/notebook'),          # 이 노트북의 모든 결과
    'cache_dir': str(PKG / 'runs/baseline/cache'),    # 특징 캐시. 키가 같으면 다시 계산하지 않음
    'psd_file':  str(PKG / 'runs/notebook/psd_features.npz'),          # 없으면 계산해서 저장
    'fullrot_file': str(PKG / 'runs/notebook/fullrot_features.npz'),  # 셀 3용, 없으면 계산
    # --- 실행 ---
    'train': True,            # False면 학습 없이 ckpt_dir의 체크포인트로 평가만
    'ckpt_dir': None,         # None이면 out_dir. 예: str(PKG/'runs/baseline') 로 기존 결과 재평가
    'seeds': [42, 43, 44],
    'init_offset': 0,         # 분할은 seed 그대로, 가중치 초기화/셔플 seed만 바꿀 때
    'tasks': ['ssl', 'id'],
    'ssl_target': 'fault',    # 'fault': 고장 소리만 찾기 (개수 0~3, 고장 위치 각도). 'all': 기존(3음원 전부, 정답 항상 같음)
    'device': 'cuda',
    'gpu_duty': 0.5,          # GPU 평균 사용률 목표(발열 감소). 0.5 → 약 50%, 학습 시간 약 2배. 1.0 = 제한 없음
    # --- 특징 (configs/ev.json 과 같음) ---
    'sample_rate': 48000, 'n_fft': 2048, 'hop_length': 480, 'n_mels': 64,
    'fmin': 0.0, 'fmax': 12000.0, 'db_floor': -120.0,
    'angular_bins': 360, 'ssl_context_seconds': 1.0, 'id_frames': 100,
    'gaussian_sigma_degrees': 5.0, 'minimum_peak_spacing_degrees': 30,
    'geometry_status': 'assumed_from_recording_setup_verify_before_absolute_DOA_claims',
    'geometry': {'FRONT': 180.0, 'REAR': 0.0, 'RIGHT': 270.0},   # 미확인 기하 가정
    'rois': {'FRONT': [135.0, 225.0], 'REAR': [315.0, 405.0], 'RIGHT': [225.0, 315.0]},
    'positions': ['FRONT', 'REAR', 'RIGHT'],
    'classes': ['Healthy', 'Imbalance', 'Modulation', 'Whine', 'Wind', 'PWM'],
    'split_mode': 'recording_holdout_closed_source',
    # --- ID 모델 추가 입력 (FCN 앞에 concat) ---
    'id_task': 'binary',      # 'binary': 위치별 정상/고장만 / 'type': 6종(정상+고장 5종) 분류
    'use_rpm': False,         # 추정 RPM (측정값 아님)
    'use_psd': False,         # 위치 구간별 Welch PSD 64대역
    'psd_bands': 64,
    # --- 학습 ---
    'batch_size': 16, 'epochs': 50, 'patience': 10,
    'learning_rate': 1e-4, 'weight_decay': 1e-5,
    'location_loss_weight': 5.0, 'fault_threshold': 0.5, 'workers': 0,
}
POS = CONFIG['positions']; CLASSES = CONFIG['classes']
BINARY = CONFIG['id_task'] == 'binary'
ID_CLASSES = ['Healthy', 'Fault'] if BINARY else CLASSES      # ID 모델 출력 클래스

def id_label(c):
    return int(c != 'Healthy') if BINARY else CLASSES.index(c)

# ==========================================
# 0. 보조 함수 (파일 읽기/쓰기, 해시)
# ==========================================
def read_csv(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

def write_csv(path, rows):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda: f.read(1 << 20), b''): h.update(part)
    return h.hexdigest()

def data_path(root, relative):
    """Windows 경로 구분자 정리, 데이터 루트 밖으로 못 나가게 막음."""
    root = Path(root).resolve(); relative = str(relative).replace('\\', '/')
    if relative.startswith('/') or ':' in relative or '..' in Path(relative).parts:
        raise ValueError('Expected a relative dataset path: ' + relative)
    path = (root / relative).resolve()
    if not path.is_relative_to(root): raise ValueError('Dataset path escaped root.')
    return path

def records_by_ids(records, ids):
    lookup = {r['uid']: r for r in records}
    return [lookup[u] for u in ids]

# ==========================================
# 2. Parsing Dataset (측정_목록.csv + 녹음별 info CSV)
# ==========================================
def interval(text):
    a, b = map(float, text.split('~'))
    if not np.isfinite([a, b]).all() or b < a: raise ValueError('Invalid interval: ' + text)
    return [a, b]

def discover(root):
    """3대 스피커 녹음만 읽는다(1대 녹음 200개 제외). 라벨/개수 불일치는 바로 멈춘다."""
    root = Path(root).resolve(); records = []
    for row in read_csv(root / '측정_목록.csv'):
        if int(row['스피커수']) != 3: continue
        paths = {k: str(row[c]).replace('\\', '/') for k, c in
                 [('wav', 'WAV'), ('info', '실험정보CSV'), ('angles', '각도CSV')]}
        for rel in paths.values():
            if not data_path(root, rel).is_file(): raise FileNotFoundError(rel)
        info = read_csv(data_path(root, paths['info']))
        if len(info) != 3 or {s['Speaker_Loc'] for s in info} != set(POS):
            raise ValueError('Expected exactly three unique positions: ' + paths['info'])
        src = {s['Speaker_Loc']: s for s in info}; wav_name = Path(paths['wav']).name
        for pos, s in src.items():
            if s['status'] != 'OK' or s['Saved_Filename'] != wav_name:
                raise ValueError('Non-successful or inconsistent capture: ' + paths['info'])
            if s['Sound_Class'] not in CLASSES: raise ValueError(s['Sound_Class'])
            if s['Sound_Class'] != row[{'FRONT': '앞', 'REAR': '뒤', 'RIGHT': '오른쪽'}[pos]]:
                raise ValueError('Index/source label mismatch.')
        fault_count = sum(s['Sound_Class'] != 'Healthy' for s in info)
        if fault_count != int(row['고장음위치수']): raise ValueError('Fault count mismatch.')
        duration = float(info[0]['rotation_time_sec'])
        if not 3.7 <= duration <= 4.3: raise ValueError('Expected encoder-gated ~4 s audio.')
        rec = dict(uid=row['측정ID'], **paths, band=row['추정RPM구간'], fault_count=fault_count,
                   source_count=3, previous_role=row['학습분할'], original_order=int(row['원본회차']),
                   duration=duration, classes=[src[p]['Sound_Class'] for p in POS],
                   source_files=[src[p]['Source_File'].replace('\\', '/') for p in POS],
                   rpm_intervals=[interval(src[p]['Estimated_RPM_Interval']) for p in POS],
                   event_time=info[0]['event_time'])
        rec['input_fingerprint'] = {k: dict(bytes=data_path(root, v).stat().st_size,
                                            sha256=digest(data_path(root, v))) for k, v in paths.items()}
        records.append(rec)
    records.sort(key=lambda r: r['uid'])
    if len({r['uid'] for r in records}) != len(records): raise ValueError('Duplicate recording UID.')
    return records

def make_split(records, seed):
    """고장 위치 수 × RPM 구간 층화, 녹음 단위 8:1:1 (closed-source recording holdout)."""
    groups = defaultdict(list)
    for r in records: groups[(r['fault_count'], r['band'])].append(r['uid'])
    rng = np.random.default_rng(seed)
    split = dict(seed=seed, mode='recording_holdout_closed_source', train=[], validation=[], test=[])
    for key in sorted(groups):
        ids = np.array(sorted(groups[key]), dtype=object); rng.shuffle(ids)
        n = len(ids); a = int(.8 * n); b = int(.1 * n)
        if min(a, b, n - a - b) < 1: raise ValueError('Insufficient stratum: ' + str(key))
        split['train'] += ids[:a].tolist(); split['validation'] += ids[a:a + b].tolist()
        split['test'] += ids[a + b:].tolist()
    for k in ['train', 'validation', 'test']: split[k].sort()
    sets = [set(split[k]) for k in ['train', 'validation', 'test']]
    if any(sets[i] & sets[j] for i in range(3) for j in range(i)): raise AssertionError('Recording leakage.')
    lookup = {r['uid']: r for r in records}
    src = {k: {s for u in split[k] for s in lookup[u]['source_files']} for k in ['train', 'validation', 'test']}
    split['source_file_overlap'] = {a + '__' + b: len(src[a] & src[b])
        for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]}
    split['unseen_source_generalization_claim_allowed'] = False
    return split

# ==========================================
# 3. Feature Extraction (Log-Mel, 각도 정렬, ROI 잘라내기, PSD)
# ==========================================
def mel_bank():
    c = CONFIG
    hz2mel = lambda x: 2595.0 * np.log10(1.0 + np.asarray(x) / 700.0)
    mel2hz = lambda x: 700.0 * (10.0 ** (np.asarray(x) / 2595.0) - 1.0)
    pts = mel2hz(np.linspace(hz2mel(c['fmin']), hz2mel(c['fmax']), c['n_mels'] + 2))
    f = np.fft.rfftfreq(c['n_fft'], 1.0 / c['sample_rate'])
    l = (f[None] - pts[:-2, None]) / (pts[1:-1, None] - pts[:-2, None])
    r = (pts[2:, None] - f[None]) / (pts[2:, None] - pts[1:-1, None])
    w = np.maximum(0.0, np.minimum(l, r)); w /= np.maximum(w.sum(1, keepdims=True), 1e-12)
    return w.astype(np.float32)

def logmel(audio):
    c = CONFIG; n = c['n_fft']
    if len(audio) < n: audio = np.pad(audio, (0, n - len(audio)))
    _, times, power = signal.spectrogram(audio.astype(np.float64), fs=c['sample_rate'], window='hann',
        nperseg=n, noverlap=n - c['hop_length'], detrend=False, scaling='spectrum', mode='psd')
    db = 10.0 * np.log10(np.maximum(mel_bank() @ power, 10.0 ** (c['db_floor'] / 10.0)))
    return db.astype(np.float32), times

def angle_track(path, duration):
    """각도 CSV → (시간, 각도) 단조 증가 쌍. 불연속은 고치지 않고 멈춘다."""
    rows = read_csv(path)
    t = np.array([float(r['elapsed_sec']) for r in rows]); a = np.array([float(r['angle_deg']) for r in rows])
    keep = np.isfinite(t) & np.isfinite(a) & (t >= 0) & (t <= duration); t, a = t[keep], a[keep]
    if len(t) < 20 or np.any(np.diff(t) < 0): raise ValueError('Invalid angle timestamps.')
    t, u = np.unique(t, return_index=True); a = a[u]
    if np.min(a) < 0 or np.max(a) > 360: raise ValueError('Angles outside 0..360.')
    if np.any(np.diff(a) < -1.0) or np.any(np.diff(a) > 20.0):
        raise ValueError('Unacceptable recorded angle discontinuity; no silent repair.')
    inc = np.r_[True, a[1:] > np.maximum.accumulate(a[:-1])]; t, a = t[inc], a[inc]
    interior = (t > 0) & (t < duration) & (a > 0) & (a < 360)
    return np.r_[0.0, t[interior], duration], np.r_[0.0, a[interior], 360.0]

def circular_segments(start, end):
    """뒤 구간 315–405°는 315–360 / 0–45 두 조각으로 분리 (경계 넘어 이어 붙이지 않음)."""
    if not 0 <= start < 360 or not start < end <= start + 360: raise ValueError('Invalid angular ROI.')
    return [(start, end)] if end <= 360 else [(start, 360.0), (0.0, end - 360.0)]

def ssl_input(db, times, at, aa):
    """360칸 각도별 Log-Mel (중심 프레임 + 1초 문맥 평균) → (2*n_mels, 360)."""
    c = CONFIG
    centers = np.interp(np.arange(360), aa, at)
    central = np.stack([np.interp(centers, times, band) for band in db])
    ctx = float(c['ssl_context_seconds'])
    if ctx <= 0: return central.astype(np.float32)
    p = np.power(10.0, db.astype(np.float64) / 10.0)
    sums = np.c_[np.zeros(len(db)), np.cumsum(p, axis=1)]
    lo = np.searchsorted(times, centers - ctx / 2, 'left'); hi = np.searchsorted(times, centers + ctx / 2, 'right')
    lo = np.minimum(lo, len(times) - 1); hi = np.maximum(hi, lo + 1)
    local = 10 * np.log10(np.maximum((sums[:, hi] - sums[:, lo]) / np.maximum(hi - lo, 1), 10 ** (c['db_floor'] / 10)))
    return np.concatenate([central, local], axis=0).astype(np.float32)

def extract_record(rec):
    c = CONFIG; root = c['data_root']
    audio, fs = sf.read(data_path(root, rec['wav']), dtype='float32')
    if fs != c['sample_rate'] or audio.ndim != 1 or not np.isfinite(audio).all():
        raise ValueError('Expected finite mono 48 kHz WAV: ' + rec['uid'])
    duration = len(audio) / fs
    if abs(duration - rec['duration']) > .002: raise ValueError('WAV/metadata duration mismatch.')
    at, aa = angle_track(data_path(root, rec['angles']), duration)
    db, times = logmel(audio)
    m, T = c['n_mels'], c['id_frames']
    views = np.full((3, 2, m, T), c['db_floor'], np.float32); masks = np.zeros((3, 2, T), np.float32)
    durs = np.zeros((3, 2), np.float32); rpm = np.zeros((3, 2), np.float32)
    common_rpm = np.median(np.array(rec['rpm_intervals']), axis=0)
    for pi, pos in enumerate(POS):
        mids, wts = [], []
        for vi, (a, b) in enumerate(circular_segments(*c['rois'][pos])):
            t0, t1 = np.interp([a, b], aa, at)
            seg = audio[round(t0 * fs):round(t1 * fs)]
            if len(seg) < 2: raise ValueError('Empty angular ROI.')
            part, _ = logmel(seg); n = part.shape[1]
            if n > T:   # 특징의 시간축만 리샘플 (원음/라벨/RPM은 그대로)
                part = np.stack([np.interp(np.linspace(0, n - 1, T), np.arange(n), v) for v in part]); n = T
            views[pi, vi, :, :n] = part; masks[pi, vi, :n] = 1; durs[pi, vi] = t1 - t0
            mids.append(np.interp((t0 + t1) / 2, [0, duration], common_rpm)); wts.append(t1 - t0)
        mean = np.average(mids, weights=wts); spread = np.sqrt(np.average((np.asarray(mids) - mean) ** 2, weights=wts))
        rpm[pi] = [mean / 10000.0, spread / 10000.0]
    return dict(ssl=ssl_input(db, times, at, aa), id_views=views, id_masks=masks, view_durations=durs, rpm=rpm)

FEATURE_KEYS = ['sample_rate', 'n_fft', 'hop_length', 'n_mels', 'fmin', 'fmax', 'db_floor',
                'angular_bins', 'ssl_context_seconds', 'id_frames', 'positions', 'rois']

def feature_key(rec):
    """ev3dar.features.feature_key 와 같은 계산 → 기존 runs/baseline/cache 재사용 가능."""
    cfg = {k: CONFIG[k] for k in FEATURE_KEYS}
    return hashlib.sha256(json.dumps(dict(version=1, uid=rec['uid'], cfg=cfg,
        inputs=rec['input_fingerprint']), sort_keys=True).encode()).hexdigest()

def cache_path(uid):
    if not uid.replace('_', '').isalnum(): raise ValueError('Invalid cache UID.')
    return Path(CONFIG['cache_dir']) / (uid + '.npz')

def build_cache(records):
    Path(CONFIG['cache_dir']).mkdir(parents=True, exist_ok=True); made = 0
    for i, r in enumerate(records, 1):
        p = cache_path(r['uid']); key = feature_key(r)
        if p.exists():
            with np.load(p, allow_pickle=False) as z:
                if str(z['feature_key'].item()) == key: continue
        tmp = p.with_suffix('.pending.npz')
        np.savez_compressed(tmp, feature_key=np.array(key), **extract_record(r)); tmp.replace(p); made += 1
        if made % 25 == 0: print(f'Features {i}/{len(records)}', flush=True)
    print(f'Feature cache ready: {len(records)} recordings ({made} newly computed)')

PSD_EDGES = np.geomspace(20, 20000, CONFIG['psd_bands'] + 1)

def psd_vec(seg, fs):
    f, p = signal.welch(seg, fs=fs, window='hann', nperseg=min(4096, len(seg)), noverlap=None)
    out = np.empty(len(PSD_EDGES) - 1)
    for i in range(len(out)):
        m = (f >= PSD_EDGES[i]) & (f < PSD_EDGES[i + 1])
        out[i] = p[m].mean() if m.any() else np.interp(np.sqrt(PSD_EDGES[i] * PSD_EDGES[i + 1]), f, p)
    return out

def extract_psd(rec):
    """위치 구간별 Welch PSD → 로그 간격 64대역(20 Hz–20 kHz), 뒤 두 조각은 길이 가중 평균 → dB (3, 64)."""
    x, fs = sf.read(data_path(CONFIG['data_root'], rec['wav']), dtype='float64')
    at, aa = angle_track(data_path(CONFIG['data_root'], rec['angles']), len(x) / fs)
    res = np.zeros((3, len(PSD_EDGES) - 1))
    for pi, pos in enumerate(POS):
        acc = 0; wsum = 0
        for a, b in circular_segments(*CONFIG['rois'][pos]):
            t0, t1 = np.interp([a, b], aa, at)
            acc = acc + psd_vec(x[round(t0 * fs):round(t1 * fs)], fs) * (t1 - t0); wsum += t1 - t0
        res[pi] = 10 * np.log10(acc / wsum + 1e-20)
    return res.astype(np.float32)

def load_or_build_psd(records):
    path = Path(CONFIG['psd_file'])
    if path.exists():
        z = np.load(path); by = {str(u): z['psd'][i] for i, u in enumerate(z['uids'])}
        if all(r['uid'] in by for r in records):
            print('PSD loaded:', path); return by
    print('Computing PSD for', len(records), 'recordings ...', flush=True)
    P = np.stack([extract_psd(r) for r in records])
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, psd=P, uids=np.array([r['uid'] for r in records]), band_edges_hz=PSD_EDGES)
    return {r['uid']: P[i] for i, r in enumerate(records)}

# ==========================================
# 4. Normalization & Dataset (train 녹음 통계만 사용)
# ==========================================
def fit_normalization(train_records, psd_by=None):
    ts = tq = None; tn = 0; is_ = iq = None; inn = 0
    for r in train_records:
        with np.load(cache_path(r['uid']), allow_pickle=False) as z:
            if str(z['feature_key'].item()) != feature_key(r): raise ValueError('Stale cache.')
            x = z['ssl'].astype(np.float64)
            if ts is None: ts = np.zeros(x.shape[0]); tq = ts.copy()
            ts += x.sum(1); tq += (x * x).sum(1); tn += x.shape[1]
            a = z['id_views'].astype(np.float64); m = z['id_masks'][:, :, None, :]
            if is_ is None: is_ = np.zeros(a.shape[2]); iq = is_.copy()
            is_ += (a * m).sum((0, 1, 3)); iq += (a * a * m).sum((0, 1, 3)); inn += m.sum()
    def stats(s, q, n):
        mean = s / n; return dict(mean=mean.tolist(), std=np.sqrt(np.maximum(q / n - mean * mean, 1.0)).tolist())
    norm = dict(ssl=stats(ts, tq, tn), id=stats(is_, iq, inn),
                fitted_recording_ids=[r['uid'] for r in train_records], train_only=True)
    if psd_by is not None:
        S = np.stack([psd_by[r['uid']] for r in train_records])
        norm['psd'] = dict(mean=S.mean((0, 1)).tolist(), std=np.maximum(S.std((0, 1)), 1e-3).tolist())
    return norm

def ssl_angles(rec, mode):
    """SSL 정답 각도. 'fault' = 고장 소리가 난 위치만 (Healthy는 어디서나 나는 배경이라 제외)."""
    return [CONFIG['geometry'][p] for p, c in zip(POS, rec['classes']) if mode != 'fault' or c != 'Healthy']

def ssl_count_label(rec, mode):
    return rec['fault_count'] if mode == 'fault' else rec['source_count'] - 1   # fault: 0~3 / all: 1~3 → 0~2

def gaussian_target(degrees, sigma=5.0):
    x = np.arange(360); target = np.zeros(360, np.float32)
    for ang in degrees:
        d = np.abs(x - (float(ang) % 360)); d = np.minimum(d, 360 - d)
        target = np.maximum(target, np.exp(-.5 * (d / sigma) ** 2))
    return target

class EVDataset(Dataset):
    def __init__(self, records, norm, psd_by=None):
        self.records, self.psd_by = records, psd_by
        self.sm = np.asarray(norm['ssl']['mean'], np.float32)[:, None]
        self.ss = np.asarray(norm['ssl']['std'], np.float32)[:, None]
        self.im = np.asarray(norm['id']['mean'], np.float32)[None, None, :, None]
        self.istd = np.asarray(norm['id']['std'], np.float32)[None, None, :, None]
        self.psd_norm = norm.get('psd')
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        with np.load(cache_path(r['uid']), allow_pickle=False) as z:
            if str(z['feature_key'].item()) != feature_key(r): raise ValueError('Stale feature cache.')
            ssl = z['ssl'].astype(np.float32); views = z['id_views'].astype(np.float32)
            masks = z['id_masks'].copy(); durs = z['view_durations'].copy(); rpm = z['rpm'].copy()
        ssl = (ssl - self.sm) / self.ss; views = ((views - self.im) / self.istd) * masks[:, :, None, :]
        if self.psd_by is not None and self.psd_norm is not None:
            psd = (self.psd_by[r['uid']] - np.asarray(self.psd_norm['mean'], np.float32)) / \
                  np.asarray(self.psd_norm['std'], np.float32)
        else:
            psd = np.zeros((3, 1), np.float32)
        return dict(uid=r['uid'], ssl=torch.from_numpy(ssl), views=torch.from_numpy(views),
                    masks=torch.from_numpy(masks), durations=torch.from_numpy(durs), rpm=torch.from_numpy(rpm),
                    psd=torch.from_numpy(psd.astype(np.float32)),
                    labels=torch.tensor([id_label(c) for c in r['classes']], dtype=torch.long),
                    count=torch.tensor(ssl_count_label(r, CONFIG['ssl_target']), dtype=torch.long),
                    target=torch.from_numpy(gaussian_target(ssl_angles(r, CONFIG['ssl_target']), CONFIG['gaussian_sigma_degrees'])))

# ==========================================
# 5. Model Architecture & Loss Functions
# ==========================================
class LocalizationCNN(nn.Module):
    """SSL: 펭귄식 원형 1D CNN → 음원 수(1~3) + 360칸 방향 점수."""
    def __init__(self, channels, n_count=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(channels, 128, 7, padding=3, padding_mode='circular'), nn.InstanceNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 5, padding=2, padding_mode='circular'), nn.InstanceNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, 5, padding=4, dilation=2, padding_mode='circular'), nn.InstanceNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 128, 3, padding=1, padding_mode='circular'), nn.InstanceNorm1d(128), nn.ReLU())
        self.count_branch = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(128, 64),
                                          nn.ReLU(), nn.Dropout(.3), nn.Linear(64, n_count))
        self.loc_branch = nn.Sequential(nn.Conv1d(128, 64, 3, padding=1, padding_mode='circular'),
                                        nn.ReLU(), nn.Conv1d(64, 1, 1))
    def forward(self, x):
        f = self.features(x); return self.count_branch(f), self.loc_branch(f).squeeze(1)

class SoundTypeCNN(nn.Module):
    """ID: 스펙트로그램 2D CNN → 벡터 (+PSD/RPM concat) → FCN → 64차원 latent → 6종 분류 decoder."""
    def __init__(self, classes=6, use_rpm=False, psd_dim=0):
        super().__init__(); self.use_rpm, self.psd_dim = use_rpm, psd_dim
        layers = []; prev = 1
        for c in [32, 64, 128]:
            layers += [nn.Conv2d(prev, c, 3, padding=1), nn.GroupNorm(8, c), nn.GELU(), nn.MaxPool2d(2)]; prev = c
        self.encoder = nn.Sequential(*layers)
        self.fcn = nn.Sequential(nn.Linear(128 + psd_dim + (2 if use_rpm else 0), 128), nn.GELU(), nn.Dropout(.2),
                                 nn.Linear(128, 64), nn.GELU())
        self.decoder = nn.Linear(64, classes)   # 분류 decoder (파형 복원 아님)
    def forward(self, views, masks, durations, rpm, psd=None):
        b, p, v, m, t = views.shape
        x = self.encoder(views.reshape(b * p * v, 1, m, t))
        valid = F.adaptive_avg_pool1d(masks.reshape(b * p * v, 1, t), x.shape[-1]).unsqueeze(2)
        z = (x * valid).sum((2, 3)) / (valid.sum((2, 3)) * x.shape[2]).clamp_min(1e-6)
        z = z.reshape(b, p, v, -1)
        w = durations * (masks.sum(-1) > 0).to(durations.dtype)          # 뒤 두 조각: 길이 가중 평균
        z = (z * w.unsqueeze(-1)).sum(2) / w.sum(2, keepdim=True).clamp_min(1e-6)
        if self.psd_dim: z = torch.cat([z, psd], dim=-1)
        if self.use_rpm: z = torch.cat([z, rpm], dim=-1)
        latent = self.fcn(z)
        return self.decoder(latent), latent

def focal_loss(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    return (.25 * (1 - torch.exp(-bce)).pow(2) * bce).mean()

def build_model(task, cfg):
    if task == 'ssl': return LocalizationCNN(cfg['n_mels'] * (2 if cfg['ssl_context_seconds'] > 0 else 1),
                                             4 if cfg.get('ssl_target', 'all') == 'fault' else 3)
    return SoundTypeCNN(2 if cfg.get('id_task', 'type') == 'binary' else len(cfg['classes']), cfg.get('use_rpm', False),
                        cfg.get('psd_bands', 64) if cfg.get('use_psd', False) else 0)

# ==========================================
# 6. Evaluation Functions
# ==========================================
def circular_peaks(prob, count, min_spacing=30):
    prob = np.asarray(prob, float)
    if count <= 0: return []                                         # 고장 0곳이면 각도 없음
    peaks, _ = find_peaks(np.tile(prob, 3), prominence=.01)          # 0/359°도 내부 후보로
    cand = sorted([int(i - 360) for i in peaks if 360 <= i < 720], key=lambda i: (-prob[i], i)); sel = []
    for c in cand:
        if all(min(abs(c - p), 360 - abs(c - p)) >= min_spacing for p in sel): sel.append(c)
        if len(sel) >= count: break
    return sel

def angular_error(pred, truth):
    truth = np.asarray(truth, float); pred = np.asarray(pred, float)
    if len(truth) == 0: return 0.0
    if len(pred) == 0: return 180.0
    cost = np.abs(truth[:, None] - pred[None]); cost = np.minimum(cost, 360 - cost)
    i, j = linear_sum_assignment(cost)
    return float((cost[i, j].sum() + 180.0 * (len(truth) - len(i))) / len(truth))   # 못 찾은 음원은 180°

def classification_metrics(labels, probs, threshold=.5):
    labels = np.asarray(labels, int).reshape(-1); probs = np.asarray(probs, float).reshape(-1, len(ID_CLASSES))
    pred = probs.argmax(-1); fault = (labels != 0).astype(int); score = 1 - probs[:, 0]
    binary = (score >= threshold).astype(int); both = len(np.unique(fault)) == 2
    return dict(n_regions=len(labels), type_accuracy=float(np.mean(pred == labels)),
        type_macro_f1=float(f1_score(labels, pred, labels=np.arange(len(ID_CLASSES)), average='macro', zero_division=0)),
        type_confusion_matrix=confusion_matrix(labels, pred, labels=np.arange(len(ID_CLASSES))).tolist(),
        fault_accuracy=float(np.mean(binary == fault)), fault_f1=float(f1_score(fault, binary, zero_division=0)),
        fault_balanced_accuracy=float(balanced_accuracy_score(fault, binary)) if both else float(np.mean(binary == fault)),
        fault_roc_auc=float(roc_auc_score(fault, score)) if both else None,
        fault_average_precision=float(average_precision_score(fault, score)) if both else None,
        fault_confusion_matrix=confusion_matrix(fault, binary, labels=[0, 1]).tolist())

# ==========================================
# 7. Train Loop
# ==========================================
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True

class GpuThrottle:
    """학습 step마다 GPU가 일한 시간에 비례해 쉬어서 평균 사용률을 gpu_duty 근처로 낮춘다.
    결과(가중치/지표)는 같고 시간만 늘어난다. GB10은 nvidia-smi 전력 제한이 안 돼서 소프트웨어로 한다."""
    def __init__(self, duty): self.duty = float(duty); self.t = time.perf_counter()
    def reset(self): self.t = time.perf_counter()
    def step(self):
        if self.duty >= 1: return
        if torch.cuda.is_available(): torch.cuda.synchronize()
        busy = time.perf_counter() - self.t
        time.sleep(busy * (1 - self.duty) / max(self.duty, 0.05)); self.t = time.perf_counter()

def make_loader(records, norm, psd_by, shuffle=False, seed=42):
    return DataLoader(EVDataset(records, norm, psd_by), batch_size=CONFIG['batch_size'], shuffle=shuffle,
                      num_workers=CONFIG['workers'], generator=torch.Generator().manual_seed(seed),
                      pin_memory=torch.cuda.is_available())

def forward(model, task, batch, device):
    if task == 'ssl': return model(batch['ssl'].to(device, non_blocking=True))
    return model(*(batch[k].to(device, non_blocking=True) for k in ['views', 'masks', 'durations', 'rpm', 'psd']))

def ssl_loss(counts, loc, batch, device):
    return F.cross_entropy(counts, batch['count'].to(device)) + \
           CONFIG['location_loss_weight'] * focal_loss(loc, batch['target'].to(device))

def get_device():
    if CONFIG['device'] == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable. Run on the Spark GPU; no automatic CPU fallback.')
    return torch.device(CONFIG['device'])

def validate(model, task, loader, device):
    model.eval(); loss_sum = 0.0; Y = []; P = []
    with torch.no_grad():
        for b in loader:
            a, c = forward(model, task, b, device)
            if task == 'ssl': loss_sum += float(ssl_loss(a, c, b, device)) * len(b['uid'])
            else: Y.append(b['labels'].numpy()); P.append(a.softmax(-1).cpu().numpy())
    if task == 'ssl':
        loss = loss_sum / len(loader.dataset); return dict(loss=loss), -loss
    m = classification_metrics(np.concatenate(Y), np.concatenate(P), CONFIG['fault_threshold'])
    return m, m['type_macro_f1']            # ID는 validation macro-F1로 best 선택

def train_task(task, seed, split, train_recs, valid_recs, norm, psd_by, run_dir):
    device = get_device(); seed_all(seed + CONFIG['init_offset'])
    tr = make_loader(train_recs, norm, psd_by, True, seed + CONFIG['init_offset'])
    va = make_loader(valid_recs, norm, psd_by)
    model = build_model(task, CONFIG).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    best = -float('inf'); stale = 0; history = []; throttle = GpuThrottle(CONFIG['gpu_duty'])
    for epoch in range(1, CONFIG['epochs'] + 1):
        model.train(); total = 0.0; n = 0; tick = time.monotonic(); throttle.reset()
        for b in tr:
            opt.zero_grad(set_to_none=True); a, c = forward(model, task, b, device)
            loss = ssl_loss(a, c, b, device) if task == 'ssl' else \
                   F.cross_entropy(a.reshape(-1, len(ID_CLASSES)), b['labels'].to(device).reshape(-1))
            if not torch.isfinite(loss): raise RuntimeError('Nonfinite loss.')
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            total += float(loss.detach()) * len(b['uid']); n += len(b['uid']); throttle.step()
        metrics, score = validate(model, task, va, device)
        history.append(dict(epoch=epoch, train_loss=total / n, selection_score=score,
                            validation={k: v for k, v in metrics.items() if 'matrix' not in k},
                            seconds=time.monotonic() - tick))
        print(f"[seed{seed} {task}] epoch {epoch:3d}  train_loss {total / n:.4f}  "
              f"val {'loss' if task == 'ssl' else 'macroF1'} {abs(score):.4f}", flush=True)
        if score > best:
            best = score; stale = 0
            ckpt = dict(state_dict={k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                        task=task, config=dict(CONFIG), normalization=norm, seed=seed, epoch=epoch,
                        validation=metrics, split=split, training_claim='closed-source held-out-recording baseline')
            tmp = run_dir / f'{task}_best.pending.pt'; torch.save(ckpt, tmp); tmp.replace(run_dir / f'{task}_best.pt')
        else:
            stale += 1
        write_json(run_dir / f'{task}_history.json', history)
        if stale >= CONFIG['patience']: break
    print(f'[seed{seed} {task}] best epoch {max(history, key=lambda h: h["selection_score"])["epoch"]}', flush=True)
    del model, opt
    if device.type == 'cuda': torch.cuda.empty_cache()

# ==========================================
# 8. Multi-Seed Loop Start!
# ==========================================
OUT = Path(CONFIG['out_dir']); OUT.mkdir(parents=True, exist_ok=True)
print('Reading dataset (read-only):', CONFIG['data_root'], flush=True)
RECORDS = discover(CONFIG['data_root'])
_groups = Counter(r['fault_count'] for r in RECORDS)
if len(RECORDS) != 1200 or _groups != {0: 300, 1: 300, 2: 300, 3: 300}:
    raise ValueError('Expected 1200 three-speaker recordings (300 per fault count): ' + str(_groups))
LOOKUP = {r['uid']: r for r in RECORDS}
SPLITS = {s: make_split(RECORDS, s) for s in CONFIG['seeds']}
(OUT / 'manifest.jsonl').write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in RECORDS), encoding='utf-8')
for s, sp in SPLITS.items(): write_json(OUT / f'splits/seed{s}.json', sp)
write_json(OUT / 'config.json', CONFIG)
print('Recordings:', len(RECORDS), dict(_groups), '| split sizes:',
      {k: len(SPLITS[CONFIG['seeds'][0]][k]) for k in ['train', 'validation', 'test']})

build_cache(RECORDS)
PSD_BY = load_or_build_psd(RECORDS) if CONFIG['use_psd'] else None

if CONFIG['train']:
    for seed in CONFIG['seeds']:
        sp = SPLITS[seed]; run_dir = OUT / f'seed{seed}'; run_dir.mkdir(exist_ok=True)
        train_recs = records_by_ids(RECORDS, sp['train']); valid_recs = records_by_ids(RECORDS, sp['validation'])
        norm = fit_normalization(train_recs, PSD_BY); write_json(run_dir / 'normalization.json', norm)
        for task in CONFIG['tasks']:
            train_task(task, seed, sp, train_recs, valid_recs, norm, PSD_BY, run_dir)
    print('Training done. Raw dataset was read only.')

In [ ]:
# ==========================================
# 1. Test Evaluation (seed별 → test_predictions.csv / test_metrics.json)
# ==========================================
def load_ckpt(path, task, device):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)   # 이 프로젝트가 만든 파일만 로드
    if ckpt['task'] != task: raise ValueError('Checkpoint task mismatch.')
    model = build_model(task, ckpt['config']); model.load_state_dict(ckpt['state_dict'])
    return model.to(device).eval(), ckpt

def evaluate(seed):
    device = get_device(); ck_dir = Path(CONFIG['ckpt_dir'] or CONFIG['out_dir']) / f'seed{seed}'
    out_dir = OUT / f'seed{seed}'; out_dir.mkdir(parents=True, exist_ok=True)
    id_model, ic = load_ckpt(ck_dir / 'id_best.pt', 'id', device)
    has_ssl = (ck_dir / 'ssl_best.pt').exists()
    if has_ssl:
        ssl_model, sc = load_ckpt(ck_dir / 'ssl_best.pt', 'ssl', device)
        if sc['split']['test'] != ic['split']['test']: raise ValueError('SSL/ID split mismatch.')
    if ic['split']['test'] != SPLITS[seed]['test']: raise ValueError('Checkpoint split differs from this notebook split.')
    test = records_by_ids(RECORDS, ic['split']['test'])
    psd_by = PSD_BY if ic['config'].get('use_psd') else None
    if ic['config'].get('use_psd') and psd_by is None: psd_by = load_or_build_psd(RECORDS)
    loader = make_loader(test, ic['normalization'], psd_by)
    mode = sc['config'].get('ssl_target', 'all') if has_ssl else None   # 기존 체크포인트는 'all'
    rows = []; Y = []; P = []; maes = []; count_ok = 0; pos_exact = defaultdict(list)
    with torch.no_grad():
        for b in loader:
            ip = forward(id_model, 'id', b, device)[0].softmax(-1).cpu().numpy()
            yy = b['labels'].numpy(); Y.append(yy); P.append(ip)
            if has_ssl:
                cnt, loc = forward(ssl_model, 'ssl', b, device)
                dp = loc.sigmoid().cpu().numpy(); predn = cnt.argmax(-1).cpu().numpy() + (0 if mode == 'fault' else 1)
            for j, uid in enumerate(b['uid']):
                peaks = []
                if has_ssl:
                    peaks = circular_peaks(dp[j], int(predn[j]), CONFIG['minimum_peak_spacing_degrees'])
                    truth = ssl_angles(LOOKUP[uid], mode)
                    if truth: maes.append(angular_error(peaks, truth))       # 고장 0곳 녹음은 각도 오차 없음
                    count_ok += int(predn[j] == len(truth))
                ssl_pos = []
                for k, pos in enumerate(POS):
                    a, bb = CONFIG['rois'][pos]
                    inside = [p for p in peaks if (a <= p < bb if bb <= 360 else p >= a or p < bb - 360)]
                    ssl_pos.append(int(bool(inside)))
                    fp = float(1 - ip[j, k, 0])
                    rows.append(dict(recording_id=uid, seed=seed, position=pos, fault_count=LOOKUP[uid]['fault_count'],
                        predicted_type=ID_CLASSES[int(ip[j, k].argmax())], gt_type=LOOKUP[uid]['classes'][k],
                        predicted_fault=int(fp >= CONFIG['fault_threshold']), gt_fault=int(yy[j, k] != 0),
                        fault_probability=fp,
                        predicted_angle_deg=max(inside, key=lambda p: dp[j, p]) if inside else None,
                        predicted_source_count=int(predn[j]) if has_ssl else None,
                        ssl_predicted_fault=int(bool(inside)) if has_ssl and mode == 'fault' else None,
                        band=LOOKUP[uid]['band'], **{'prob_' + c: float(ip[j, k, ci]) for ci, c in enumerate(ID_CLASSES)}))
                if has_ssl and mode == 'fault':   # 세 위치(앞/뒤/오른쪽)의 고장 여부를 SSL이 모두 맞혔나
                    gt = [int(c != 'Healthy') for c in LOOKUP[uid]['classes']]
                    pos_exact[LOOKUP[uid]['fault_count']].append(ssl_pos == gt)
    Yc = np.concatenate(Y); Pc = np.concatenate(P)
    m = classification_metrics(Yc, Pc, CONFIG['fault_threshold'])
    # train 분포만으로 맞히는 기준선 (소리를 안 듣고 위치별 클래스 비율로 예측)
    prior = np.ones((3, len(ID_CLASSES)))
    for r in records_by_ids(RECORDS, ic['split']['train']):
        for k, c in enumerate(r['classes']): prior[k, id_label(c)] += 1
    prior /= prior.sum(-1, keepdims=True)
    m['training_position_prior_baseline'] = classification_metrics(Yc, np.broadcast_to(prior, Pc.shape), CONFIG['fault_threshold'])
    m['per_position'] = {p: classification_metrics(Yc[:, k], Pc[:, k], CONFIG['fault_threshold']) for k, p in enumerate(POS)}
    if has_ssl:
        m.update(ssl_target=mode, ssl_count_accuracy=count_ok / len(test), ssl_mae_degrees=float(np.mean(maes)))
        if mode == 'fault':
            m['ssl_position_exact'] = float(np.mean([v for vs in pos_exact.values() for v in vs]))
            m.update({f'ssl_position_exact_{k}fault': float(np.mean(v)) for k, v in sorted(pos_exact.items())})
        else:
            m['ssl_labels_constant'] = True   # 모든 녹음이 3음원·같은 배치 → SSL 정답이 상수
    m.update(seed=seed, n_recordings=len(test), id_best_epoch=ic['epoch'],
             source_file_overlap=ic['split']['source_file_overlap'],
             generalization_claim='Known replay-source recordings only; NOT unseen-source localization.')
    write_json(out_dir / 'test_metrics.json', m); write_csv(out_dir / 'test_predictions.csv', rows)
    return m, rows

# ==========================================
# 2. 위치 판별 지표 (11/16 compare.py 와 같은 정의)
# ==========================================
def location_metrics(rows):
    """exact1: 고장 1곳 녹음에서 세 위치의 정상/고장 판정이 모두 맞은 비율.
       other1: 고장 1곳 녹음에서 정상 위치를 고장으로 오경보한 녹음 비율.
       FA{k}: 고장 k곳 녹음의 정상 위치 오경보율. miss: 고장 위치를 정상으로 놓친 비율.
       (RESULTS.md 표의 '고장 1곳: 정상 위치 오경보'는 FA1 이다.)"""
    by = defaultdict(list)
    for r in rows: by[r['recording_id']].append(r)
    out = {}
    for k in range(3):
        h = [r for v in by.values() if v[0]['fault_count'] == k for r in v if not r['gt_fault']]
        out[f'FA{k}'] = float(np.mean([r['predicted_fault'] for r in h])) if h else float('nan')
    one = [v for v in by.values() if v[0]['fault_count'] == 1]
    out['exact1'] = float(np.mean([all(r['predicted_fault'] == r['gt_fault'] for r in v) for v in one]))
    out['other1'] = float(np.mean([any(r['predicted_fault'] and not r['gt_fault'] for r in v) for v in one]))
    out['miss'] = float(np.mean([1 - r['predicted_fault'] for r in rows if r['gt_fault']]))
    for c in ([] if BINARY else CLASSES[1:]):
        R = [r for r in rows if r['gt_type'] == c]; out['recall_' + c] = float(np.mean([r['predicted_type'] == c for r in R]))
    return out

# ==========================================
# 3. Multi-Seed Evaluation & Grand Summary (Mean ± Std)
# ==========================================
ALL_METRICS, ALL_ROWS, LOC_METRICS = {}, [], {}
for seed in CONFIG['seeds']:
    m, rows = evaluate(seed); ALL_METRICS[seed] = m; ALL_ROWS += rows; LOC_METRICS[seed] = location_metrics(rows)

KEYS = ['type_accuracy', 'type_macro_f1', 'fault_f1', 'fault_balanced_accuracy', 'fault_roc_auc',
        'ssl_count_accuracy', 'ssl_mae_degrees', 'ssl_position_exact'] + [f'ssl_position_exact_{k}fault' for k in range(4)]
LOC_KEYS = ['exact1', 'other1', 'miss', 'FA0', 'FA1', 'FA2'] + ['recall_' + c for c in ([] if BINARY else CLASSES[1:])]
SUMMARY = {}
print(f"{'metric':34s}" + ''.join(f'seed{s:<8d}' for s in CONFIG['seeds']) + 'mean ± std')
for k in KEYS + LOC_KEYS:
    vals = [(LOC_METRICS[s] if k in LOC_KEYS else ALL_METRICS[s]).get(k) for s in CONFIG['seeds']]
    if k == 'ssl_mae_degrees' and ALL_METRICS[CONFIG['seeds'][0]].get('ssl_target') == 'fault': k = 'ssl_fault_mae_degrees'
    if any(v is None for v in vals): continue
    mean = float(np.mean(vals)); std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    SUMMARY[k] = dict(mean=mean, std=std, per_seed=vals)
    print(f'{k:34s}' + ''.join(f'{v:<12.3f}' for v in vals) + f'{mean:.3f} ± {std:.3f}')
prior_f1 = np.mean([ALL_METRICS[s]['training_position_prior_baseline']['type_macro_f1'] for s in CONFIG['seeds']])
print(f'\n(참고) 소리 없이 위치별 train 분포로만 맞힌 macroF1: {prior_f1:.3f}')
write_json(OUT / 'three_seed_summary.json', dict(seeds=CONFIG['seeds'], protocol=CONFIG['split_mode'],
    summary=SUMMARY, config=CONFIG,
    warning='No unseen-source generalization claim (closed-source recording holdout).'))

In [ ]:
# ==========================================
# SSL 결과 그림 (펭귄 셀 2와 같은 Nature style: 각도 0~360° × 확률, 정답 점선 + 예측 실선 + 봉우리 ▼)
# test 녹음마다 PDF 1장: OUT/Test_All_Plots/seed{N}/Test_Nature_{k}fault_{uid}.pdf
# ==========================================
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'], 'figure.dpi': 300})

def ssl_test_outputs(seed):
    """test 녹음별 SSL 360° 확률, 예측 고장 개수, 고른 봉우리, 정답 가우시안 지도."""
    device = get_device(); ck_dir = Path(CONFIG['ckpt_dir'] or CONFIG['out_dir']) / f'seed{seed}'
    ssl_model, sc = load_ckpt(ck_dir / 'ssl_best.pt', 'ssl', device)
    mode = sc['config'].get('ssl_target', 'all')
    recs = records_by_ids(RECORDS, sc['split']['test']); out = []
    with torch.no_grad():
        for b in make_loader(recs, sc['normalization'], None):
            cnt, loc = forward(ssl_model, 'ssl', b, device)
            dp = loc.sigmoid().cpu().numpy(); predn = cnt.argmax(-1).cpu().numpy() + (0 if mode == 'fault' else 1)
            for j, uid in enumerate(b['uid']):
                out.append(dict(uid=uid, prob=dp[j], n=int(predn[j]),
                    peaks=circular_peaks(dp[j], int(predn[j]), CONFIG['minimum_peak_spacing_degrees']),
                    target=gaussian_target(ssl_angles(LOOKUP[uid], mode), CONFIG['gaussian_sigma_degrees'])))
    return out

def plot_nature(ax, item):
    ax.plot(item['target'], color='#C44E52', linestyle='--', linewidth=1.0, label='Ground truth')
    ax.plot(item['prob'], color='#00A087', linewidth=1.2, label='Prediction')
    for p in item['peaks']:
        ax.scatter(p, item['prob'][p] + 0.05, marker='v', color='#55A868', s=20, zorder=5)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 360); ax.set_xticks(np.arange(0, 361, 60))
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0]); ax.set_ylim(0, 1.0)
    ax.set_xlabel('Angle (degree)'); ax.set_ylabel('Probability')

for seed in CONFIG['seeds']:
    plot_dir = OUT / 'Test_All_Plots' / f'seed{seed}'; plot_dir.mkdir(parents=True, exist_ok=True)
    items = ssl_test_outputs(seed)
    for item in items:
        fig, ax = plt.subplots(figsize=(4, 3), dpi=300); plot_nature(ax, item); plt.tight_layout()
        plt.savefig(plot_dir / f"Test_Nature_{LOOKUP[item['uid']]['fault_count']}fault_{item['uid']}.pdf"); plt.close(fig)
    print(f'seed{seed}: {len(items)} plots →', plot_dir)

# 화면 확인용: 첫 seed에서 고장 0/1/2/3곳 녹음 하나씩
items = {i['uid']: i for i in ssl_test_outputs(CONFIG['seeds'][0])}
picks = [next(u for u in items if LOOKUP[u]['fault_count'] == k) for k in range(4)]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4), dpi=150)
for ax, u in zip(axes, picks):
    plot_nature(ax, items[u])
    truth = ', '.join(f'{p} {c}' for p, c in zip(POS, LOOKUP[u]['classes']) if c != 'Healthy') or 'all Healthy'
    ax.set_title(f'{u}\n{truth}', fontsize=10)
axes[0].legend(frameon=False, loc='upper right', fontsize=9)
plt.tight_layout(); plt.savefig(OUT / 'Test_All_Plots' / 'examples.png'); plt.show()

In [ ]:
# ==========================================
# SSL + ID 합치기 → 최종 "고장 위치 + 각도"
# ==========================================
# 위치(앞/뒤/오른쪽)마다
#   s_ssl = SSL이 (개수만큼 고른) 피크를 이 구간에서 찾았으면 1, 아니면 구간 최대 확률 × 0.5,  s_id = ID 고장 확률 (1 - P(Healthy))
#   score = w · s_ssl + (1 - w) · s_id   → score ≥ t 이면 고장
#   고장이면 각도 = 그 구간 안 SSL 확률이 가장 큰 각도
# w, t 는 validation 녹음에서 "세 위치 정상/고장을 모두 맞힌 녹음 비율"이 최대가 되게 고르고, test 는 한 번만 본다.
# w = 1 은 SSL 만, w = 0 은 ID 만 쓴 것과 같다 (같은 방식으로 t 를 골라 비교).
def roi_mask(pos):
    a, b = CONFIG['rois'][pos]; th = np.arange(360)
    return ((th >= a) & (th < b)) if b <= 360 else ((th >= a) | (th < b - 360))
ROI_MASKS = np.stack([roi_mask(p) for p in POS])          # (3, 360)

def collect(seed, part):
    """한 분할의 녹음별·위치별 SSL 점수, SSL 각도, ID 고장 확률, 정답."""
    device = get_device(); ck_dir = Path(CONFIG['ckpt_dir'] or CONFIG['out_dir']) / f'seed{seed}'
    id_model, ic = load_ckpt(ck_dir / 'id_best.pt', 'id', device)
    ssl_model, sc = load_ckpt(ck_dir / 'ssl_best.pt', 'ssl', device)
    if sc['config'].get('ssl_target', 'all') != 'fault': raise ValueError('SSL must be trained with ssl_target="fault".')
    psd_by = load_or_build_psd(RECORDS) if ic['config'].get('use_psd') else None
    recs = records_by_ids(RECORDS, ic['split'][part])
    out = dict(uid=[], s_ssl=[], ang=[], s_id=[], gt=[])
    with torch.no_grad():
        for b in make_loader(recs, ic['normalization'], psd_by):
            pid = 1 - forward(id_model, 'id', b, device)[0].softmax(-1)[..., 0].cpu().numpy()      # (B, 3)
            cnt, loc = forward(ssl_model, 'ssl', b, device)
            dp = loc.sigmoid().cpu().numpy(); predn = cnt.argmax(-1).cpu().numpy()                 # (B, 360), 고장 개수
            for j, uid in enumerate(b['uid']):
                masked = np.where(ROI_MASKS, dp[j][None], -1)
                peaks = circular_peaks(dp[j], int(predn[j]), CONFIG['minimum_peak_spacing_degrees'])
                # SSL 점수 = 개수 head가 정한 피크가 이 구간에 있으면 1, 없으면 그 구간 최대 확률의 절반 (0~0.5)
                flag = np.array([any(ROI_MASKS[k, p] for p in peaks) for k in range(3)])
                out['uid'].append(uid); out['s_ssl'].append(np.where(flag, 1.0, 0.5 * masked.max(1))); out['ang'].append(masked.argmax(1))
                out['s_id'].append(pid[j]); out['gt'].append([int(c != 'Healthy') for c in LOOKUP[uid]['classes']])
    return {k: np.array(v) for k, v in out.items()}

def fused_eval(d, w, t):
    pred = (w * d['s_ssl'] + (1 - w) * d['s_id'] >= t).astype(int)
    exact = (pred == d['gt']).all(1); fc = d['gt'].sum(1)
    hit = (pred == 1) & (d['gt'] == 1)
    err = np.abs(d['ang'][hit] - np.array([CONFIG['geometry'][p] for p in POS])[np.where(hit)[1]])
    return dict(position_accuracy=float((pred == d['gt']).mean()), exact=float(exact.mean()),
                **{f'exact_{k}fault': float(exact[fc == k].mean()) for k in range(4)},
                missed_faults=float(((pred == 0) & (d['gt'] == 1)).sum() / max(d['gt'].sum(), 1)),
                false_alarms=float(((pred == 1) & (d['gt'] == 0)).sum() / max((d['gt'] == 0).sum(), 1)),
                angle_mae_deg=float(np.minimum(err, 360 - err).mean()) if hit.any() else None), pred

T_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)
FUSION = {}
for seed in CONFIG['seeds']:
    val, test = collect(seed, 'validation'), collect(seed, 'test')
    res = {}
    for name, ws in [('SSL only', [1.0]), ('ID only', [0.0]), ('SSL + ID', [0.25, 0.5, 0.75])]:
        w, t = max(((w, t) for w in ws for t in T_GRID), key=lambda wt: (fused_eval(val, *wt)[0]['exact'], -abs(wt[0] - .5)))
        m, pred = fused_eval(test, w, t); res[name] = dict(w=w, t=float(t), **m)
        if name == 'SSL + ID':      # 최종 출력: 녹음별 고장 위치 + 각도
            write_csv(OUT / f'seed{seed}/final_fault_locations.csv', [dict(recording_id=u, position=p,
                predicted_fault=int(pred[i, k]), angle_deg=int(test['ang'][i, k]) if pred[i, k] else None,
                score=float(w * test['s_ssl'][i, k] + (1 - w) * test['s_id'][i, k]), gt_fault=int(test['gt'][i, k]))
                for i, u in enumerate(test['uid']) for k, p in enumerate(POS)])
    FUSION[seed] = res

cols = ['position_accuracy', 'exact', 'exact_0fault', 'exact_1fault', 'exact_2fault', 'exact_3fault',
        'missed_faults', 'false_alarms', 'angle_mae_deg']
print(f"{'test, mean of seeds':22s}" + ''.join(f'{c:>14s}' for c in cols))
for name in ['SSL only', 'ID only', 'SSL + ID']:
    vals = [np.mean([FUSION[s][name][c] for s in CONFIG['seeds'] if FUSION[s][name][c] is not None]) for c in cols]
    print(f'{name:22s}' + ''.join(f'{v:14.3f}' for v in vals))
print('chosen (w, t) per seed:', {s: (FUSION[s]['SSL + ID']['w'], FUSION[s]['SSL + ID']['t']) for s in CONFIG['seeds']})
write_json(OUT / 'fusion_summary.json', FUSION)

In [ ]:
# ==========================================
# 규칙 기반 위치 판별 (학습 없음, 11/16 실험 1)
# ==========================================
# 녹음 전체 회전 → (160 mel × 90 각도칸) 특징
# R = train 의 고장 0곳 녹음 평균 (공통 무늬), D = X - R, D' = D - 각도 평균
# score(위치) = Σ_f w_f · mean_{θ∈ROI} D'_f(θ),  w_f = D 의 각도 평균 (해당 대역만)
# 대역은 train 고장 1곳 녹음 정확도로 고르고, test 는 한 번만 보고한다.
FR_NFFT, FR_NMEL, FR_FMIN, FR_FMAX, FR_NANG = 4096, 160, 50.0, 20000.0, 90

def fullrot_mel_bank():
    hz2mel = lambda x: 2595 * np.log10(1 + np.asarray(x) / 700); mel2hz = lambda x: 700 * (10 ** (np.asarray(x) / 2595) - 1)
    pts = mel2hz(np.linspace(hz2mel(FR_FMIN), hz2mel(FR_FMAX), FR_NMEL + 2))
    f = np.fft.rfftfreq(FR_NFFT, 1 / CONFIG['sample_rate'])
    l = (f[None] - pts[:-2, None]) / (pts[1:-1, None] - pts[:-2, None]); r = (pts[2:, None] - f[None]) / (pts[2:, None] - pts[1:-1, None])
    w = np.maximum(0, np.minimum(l, r))
    for i in range(FR_NMEL):
        if w[i].sum() == 0: w[i, np.argmin(abs(f - pts[i + 1]))] = 1    # 좁은 저역 대역도 최소 1개 FFT bin
    return w / w.sum(1, keepdims=True), pts[1:-1]

def extract_fullrot(rec, W):
    x, fs = sf.read(data_path(CONFIG['data_root'], rec['wav']), dtype='float64')
    at, aa = angle_track(data_path(CONFIG['data_root'], rec['angles']), len(x) / fs)
    _, t, Pw = signal.spectrogram(x, fs=fs, window='hann', nperseg=FR_NFFT, noverlap=FR_NFFT - CONFIG['hop_length'],
                                  detrend=False, scaling='spectrum', mode='psd')
    M = W @ Pw; b = np.clip((np.interp(t, at, aa) / (360 / FR_NANG)).astype(int), 0, FR_NANG - 1)
    S = np.zeros((FR_NMEL, FR_NANG)); n = np.zeros(FR_NANG)
    np.add.at(S.T, b, M.T); np.add.at(n, b, 1)
    if (n == 0).any():
        good = np.where(n > 0)[0]
        for k in np.where(n == 0)[0]:
            S[:, k] = S[:, good[np.argmin(np.minimum(abs(good - k), FR_NANG - abs(good - k)))]]; n[k] = 1
    return (10 * np.log10(S / n + 1e-14)).astype(np.float32)

fr_path = Path(CONFIG['fullrot_file'])
if fr_path.exists():
    z = np.load(fr_path); FR_X = z['X'].astype(np.float64); FR_FC = z['mel_centres_hz']; FR_IDX = {str(u): i for i, u in enumerate(z['uids'])}
else:
    W, FR_FC = fullrot_mel_bank(); print('Computing full-rotation features ...', flush=True)
    FR_X = np.stack([extract_fullrot(r, W) for r in RECORDS]).astype(np.float64)
    FR_IDX = {r['uid']: i for i, r in enumerate(RECORDS)}; fr_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(fr_path, X=FR_X.astype(np.float32), uids=np.array([r['uid'] for r in RECORDS]),
                        mel_centres_hz=FR_FC, angle_bin_deg=360 / FR_NANG)

FR_ANG = np.arange(FR_NANG) * (360 / FR_NANG) + 360 / FR_NANG / 2
ROI_MASK = {p: ((FR_ANG >= a) & (FR_ANG < b)) | ((FR_ANG + 360 >= a) & (FR_ANG + 360 < b)) for p, (a, b) in CONFIG['rois'].items()}
BANDS = [(50, 500), (500, 2000), (2000, 6000), (6000, 12000), (500, 12000), (1000, 8000), (2000, 12000), (50, 20000)]

def rule_scores(uids, R, band):
    m = (FR_FC >= band[0]) & (FR_FC < band[1])
    D = FR_X[[FR_IDX[u] for u in uids]] - R; Dp = D - D.mean(2, keepdims=True); w = D[:, m].mean(2)
    return np.stack([(w[:, :, None] * Dp[:, m][:, :, ROI_MASK[p]]).sum(1).mean(1) for p in POS], 1)

def rule_eval(uids, S):
    res = {}
    for k in (1, 2):
        sel = [i for i, u in enumerate(uids) if LOOKUP[u]['fault_count'] == k]; hit = 0; per = {p: [0, 0] for p in POS}
        for i in sel:
            t = np.array([c != 'Healthy' for c in LOOKUP[uids[i]]['classes']])
            ok = set(np.argsort(-S[i])[:k]) == set(np.where(t)[0]); hit += ok
            if k == 1: p = POS[int(np.where(t)[0][0])]; per[p][0] += ok; per[p][1] += 1
        res[k] = dict(correct=hit, n=len(sel), acc=hit / max(len(sel), 1), by_true_pos={p: f'{a}/{b}' for p, (a, b) in per.items()})
    return res

RULE_REPORT = {}
for seed in CONFIG['seeds']:
    sp = SPLITS[seed]
    R = FR_X[[FR_IDX[u] for u in sp['train'] if LOOKUP[u]['fault_count'] == 0]].mean(0)
    best_band = max(BANDS, key=lambda b: rule_eval(sp['train'], rule_scores(sp['train'], R, b))[1]['acc'])
    RULE_REPORT[seed] = dict(chosen_band=best_band, **{part: rule_eval(sp[part], rule_scores(sp[part], R, best_band))
                                                       for part in ('train', 'validation', 'test')})
    t = RULE_REPORT[seed]['test']
    print(f"seed{seed} band {best_band}: test 고장1곳 {t[1]['acc']:.2f} ({t[1]['correct']}/{t[1]['n']}), "
          f"고장2곳 {t[2]['acc']:.2f}  | 고장1곳 위치별 {t[1]['by_true_pos']}  (무작위 기준 1곳 0.33, 2곳 0.33)")
write_json(OUT / 'rule_localization.json', RULE_REPORT)

In [ ]:
# ==========================================
# 그림: 6종 혼동 행렬 (3 seed 합산, 행 정규화) + seed별 주요 지표
# ==========================================
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 8, 'axes.linewidth': 0.6})

cm = np.sum([np.array(ALL_METRICS[s]['type_confusion_matrix']) for s in CONFIG['seeds']], axis=0)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.6), gridspec_kw={'width_ratios': [1, 1.3]})
im = ax1.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax1.set_xticks(range(len(ID_CLASSES)), ID_CLASSES, rotation=45, ha='right'); ax1.set_yticks(range(len(ID_CLASSES)), ID_CLASSES)
for i in range(len(ID_CLASSES)):
    for j in range(len(ID_CLASSES)):
        ax1.text(j, i, f'{cmn[i, j]:.2f}\n({cm[i, j]})', ha='center', va='center', fontsize=6,
                 color='white' if cmn[i, j] > .5 else 'black')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('Actual'); ax1.set_title('ID (test, all seeds)')
fig.colorbar(im, ax=ax1, fraction=.046, pad=.04)

names = [('type_macro_f1', 'ID macroF1'), ('fault_roc_auc', 'Healthy/Fault AUC'),
         ('exact1', '1-fault: exact'), ('FA1', '1-fault: healthy-pos false alarm')]
x = np.arange(len(names)); wbar = .8 / len(CONFIG['seeds']); colors = ['#C4EBE7', '#7FDDCE', '#00A087']
for i, s in enumerate(CONFIG['seeds']):
    vals = [(LOC_METRICS[s] if k in LOC_METRICS[s] else ALL_METRICS[s])[k] for k, _ in names]
    ax2.bar(x + (i - (len(CONFIG['seeds']) - 1) / 2) * wbar, vals, wbar, label=f'seed{s}', color=colors[i % 3],
            edgecolor='black', linewidth=.4)
ax2.set_xticks(x, [n for _, n in names], rotation=20, ha='right'); ax2.set_ylim(0, 1); ax2.legend(frameon=False)
ax2.spines[['top', 'right']].set_visible(False); ax2.set_title('Test metrics per seed')
plt.tight_layout(); plt.savefig(OUT / 'result_summary.png', dpi=200); plt.show()
print('Saved:', OUT / 'result_summary.png')